# PopOut Game: MCTS vs Decision Trees

This notebook implements and compares two AI algorithms for playing the PopOut game:
- **Monte Carlo Tree Search (MCTS) with UCT**: A probabilistic search algorithm
- **Decision Trees**: A supervised learning approach

We'll analyze their performance, efficiency, and strategy differences.

## 1. Import Required Libraries

In [2]:

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from collections import defaultdict
import random
import time
from copy import deepcopy
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Import from project files
import sys
sys.path.append('.')
from logic import PopOutGame
from interface import HumanPlayer, AIPlayer

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

## 2. Implement PopOut Game Environment

The PopOut game is a combinatorial game where players take turns removing items from rows/columns until all items are removed.

In [3]:
# PopOut Game is now imported from logic.py
# The game uses the Connect 4 variant with pop mechanics

print("=== PopOut Game Information ===\n")
print("PopOutGame (imported from logic.py):")
print("- Board size: 6 rows x 7 columns")
print("- Players: Player 1 (X) and Player 2 (O)")
print("- Win condition: Connect 4 discs horizontally, vertically, or diagonally")
print("\nAvailable moves:")
print("  1. DROP: Add your disc to the top of any non-full column")
print("  2. POP: Remove one of your discs from the bottom (others fall down)")
print("\nSpecial rules:")
print("  - Simultaneous four-in-rows after a pop: popper wins")
print("  - Full board: player to move can declare a draw")
print("  - Repetition: 3x same position allows draw declaration")

# Create a sample game
sample_game = PopOutGame(rows=6, cols=7)
print("\n=== Sample Board State ===")
print(sample_game)
print()

# Show legal moves
print("Legal moves for Player 1:")
moves = sample_game.get_legal_moves()
drops = [m[1] + 1 for m in moves if m[0] == 'drop']
pops = [m[1] + 1 for m in moves if m[0] == 'pop']
print(f"  DROP columns: {drops}")
print(f"  POP columns: {pops}")

=== PopOut Game Information ===

PopOutGame (imported from logic.py):
- Board size: 6 rows x 7 columns
- Players: Player 1 (X) and Player 2 (O)
- Win condition: Connect 4 discs horizontally, vertically, or diagonally

Available moves:
  1. DROP: Add your disc to the top of any non-full column
  2. POP: Remove one of your discs from the bottom (others fall down)

Special rules:
  - Simultaneous four-in-rows after a pop: popper wins
  - Full board: player to move can declare a draw
  - Repetition: 3x same position allows draw declaration

=== Sample Board State ===
  1 2 3 4 5 6 7
  -------------
6 . . . . . . .
5 . . . . . . .
4 . . . . . . .
3 . . . . . . .
2 . . . . . . .
1 . . . . . . .
  -------------
Player 1's turn

Legal moves for Player 1:
  DROP columns: [1, 2, 3, 4, 5, 6, 7]
  POP columns: []


## 3. Monte Carlo Tree Search (MCTS) Implementation

MCTS is an adversarial search algorithm that explores the game tree using UCT (Upper Confidence Bound for Trees) as the evaluation function. We'll implement different strategies and compare their performance.

#### 3.1 Standard MCTS (Pure UCT)
Exploration via Upper Bound Confidence fórmula, random rollout  

In [ ]:
from mcts import MCTSNode

class MCTS:
    """
    Standard Monte Carlo Tree Search with UCT selection.
 
    Parameters
    ----------
    time_limit   : seconds of search per move (default 1.0)
    iterations   : hard cap on simulations (None = time-limited)
    c            : UCT exploration constant (default √2)
    """
 
    def __init__(
        self,
        time_limit: float = 1.0,
        iterations: Optional[int] = None,
        c: float = math.sqrt(2),
    ):
        self.time_limit = time_limit
        self.iterations = iterations
        self.c = c
        self.root: Optional[MCTSNode] = None
 
    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------
 
    def choose_move(self, game) -> Optional[Tuple[str, int]]:
        """Return the best move for the current player."""
        legal = game.get_legal_moves()
        if not legal:
            return None
        if len(legal) == 1:
            return legal[0]
 
        self.root = MCTSNode(game.copy())
        self._run_search()
        best = self.root.most_visited_child()
        return best.move
 
    def _run_search(self):
        start = time.time()
        n = 0
        while True:
            if self.iterations and n >= self.iterations:
                break
            if not self.iterations and time.time() - start > self.time_limit:
                break
            node = self._select(self.root)
            if not node.is_terminal():
                node = self._expand(node)
            reward = self._simulate(node)
            self._backpropagate(node, reward)
            n += 1
 
    # ------------------------------------------------------------------
    # Four MCTS phases
    # ------------------------------------------------------------------
 
    def _select(self, node: MCTSNode) -> MCTSNode:
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.best_child(self.c)
        return node
 
    def _expand(self, node: MCTSNode) -> MCTSNode:
        if node.untried:
            return node.expand()
        return node
 
    def _simulate(self, node: MCTSNode) -> float:
        """Random rollout — returns 1 if node.player wins, 0.5 draw, 0 loss."""
        state = node.state.copy()
        mover = node.player  # who we evaluate for
 
        while not state.game_over:
            moves = state.get_legal_moves()
            if not moves:
                break
            move = random.choice(moves)
            state.make_move(move[0], move[1])
 
        return self._outcome(state, mover)
 
    def _backpropagate(self, node: MCTSNode, reward: float):
    # Captura UMA vez quem iniciou a pesquisa
        root_player = self.root.state.current_player
        while node is not None:
            node.visits += 1
            # Se este nó foi jogado PELO jogador que está a pesquisar → reward direto
            # Se foi jogado PELO adversário → reward invertido
            if node.player == root_player:
                node.wins += reward
            else:
                node.wins += (1 - reward)
            node = node.parent
 
    # ------------------------------------------------------------------
 
    @staticmethod
    def _outcome(state, player: int) -> float:
        if state.winner == player:
            return 1.0
        if state.is_draw or state.winner is None:
            return 0.5
        return 0.0
 
    # ------------------------------------------------------------------
    # Diagnostics
    # ------------------------------------------------------------------
 
    def get_move_statistics(self) -> List[Dict]:
        if self.root is None:
            return []
        stats = []
        for ch in self.root.children:
            stats.append({
                "move": ch.move,
                "visits": ch.visits,
                "win_rate": ch.wins / ch.visits if ch.visits else 0.0,
                "uct": ch.uct_score(self.c),
            })
        return sorted(stats, key=lambda x: x["visits"], reverse=True)


#### 3.2 MCTS with RAPID ACTION VALUE ESTIMATION
Every move seen during a rollout is credited as if it had been played
first — the AMAF assumption.  The RAVE bias decays as node visits grow
(parameter k controls the decay rate).

In [ ]:
class MCTS_RAVE(MCTS):
    """
    MCTS with RAVE (Rapid Action Value Estimation) / AMAF.
 
    Every move seen during a rollout is credited as if it had been played
    first — the AMAF assumption.  The RAVE bias decays as node visits grow
    (parameter k controls the decay rate).
    """
 
    def __init__(
        self,
        time_limit: float = 1.0,
        iterations: Optional[int] = None,
        c: float = math.sqrt(2),
        k: float = 1000,
    ):
        super().__init__(time_limit=time_limit, iterations=iterations, c=c)
        self.k = k
 
    # ------------------------------------------------------------------
 
    def _select(self, node: MCTSNode) -> MCTSNode:
        """Select using RAVE score for child ranking."""
        while node.is_fully_expanded() and not node.is_terminal():
            node = self._best_rave_child(node)
        return node
 
    def _best_rave_child(self, node: MCTSNode) -> MCTSNode:
        return max(
            node.children,
            key=lambda ch: ch.rave_score(ch.move, c=self.c, k=self.k),
        )
 
    def _simulate(self, node: MCTSNode):
        """
        Random rollout that records which moves were played (for AMAF).
        Returns (reward, moves_played_as_list).
        """
        state = node.state.copy()
        mover = node.player
        moves_played: List[Tuple[str, int]] = []
 
        while not state.game_over:
            moves = state.get_legal_moves()
            if not moves:
                break
            move = random.choice(moves)
            moves_played.append(move)
            state.make_move(move[0], move[1])
 
        reward = self._outcome(state, mover)
        return reward, moves_played
 
    def _backpropagate(self, node: MCTSNode, payload):
        reward, moves_played = payload if isinstance(payload, tuple) else (payload, [])
        root_player = self.root.state.current_player  
        current = node
        while current is not None:
            current.visits += 1
            flipped = reward if current.player == root_player else (1 - reward)
            current.wins += flipped
            if current.parent is not None:
                for m in moves_played:
                    current.parent.rave_visits[m] += 1
                    current.parent.rave_wins[m] += flipped
            current = current.parent
 
    def _run_search(self):
        start = time.time()
        n = 0
        while True:
            if self.iterations and n >= self.iterations:
                break
            if not self.iterations and time.time() - start > self.time_limit:
                break
            node = self._select(self.root)
            if not node.is_terminal():
                node = self._expand(node)
            payload = self._simulate(node)       # returns (reward, moves)
            self._backpropagate(node, payload)
            n += 1
 

#### 3.3 Top-K MCTS
Branch pruning — expands only the k=7 most promising moves (out of 14)

In [ ]:
class MCTSTopK(MCTS):
    """
    Top-K MCTS: during expansion only the K children with the highest visit
    counts are kept; others are pruned.  This concentrates computation on
    the most promising branches.
 
    With 14 possible moves (7 drop + 7 pop), k=7 keeps roughly half.
    """
 
    def __init__(
        self,
        time_limit: float = 1.0,
        iterations: Optional[int] = None,
        c: float = math.sqrt(2),
        k: int = 7,
    ):
        super().__init__(time_limit=time_limit, iterations=iterations, c=c)
        self.k = k
 
    # ------------------------------------------------------------------
 
    def _expand(self, node: MCTSNode) -> MCTSNode:
        child = super()._expand(node)
        # After expansion, prune to top-k children by visits
        if len(node.children) > self.k:
            node.children.sort(key=lambda ch: ch.uct_score(self.c), reverse=True)
            node.children = node.children[:self.k]
        return child
 
    def _select(self, node: MCTSNode) -> MCTSNode:  
        """UCT selection restricted to the kept children."""
        while node.is_fully_expanded() and not node.is_terminal():
            if not node.children:
                break
            node = node.best_child(self.c)
        return node

#### 3.4 Heuristic MCTS
Biased rollout: instead of uniform random, moves are sampled according to the heuristic weight
Prior score: newly expanded nodes receive a small prior win count proportional to their heuristic value

In [ ]:
class MCTSWithHeuristics(MCTS):
    """
    Heuristic MCTS — two enhancements over standard MCTS:
 
    a) **Biased rollout**: instead of uniform random, moves are sampled
       according to a heuristic weight.  Winning moves are always taken;
       blocking moves get high weight.
 
    b) **Prior scores**: newly expanded nodes receive a small prior win
       count proportional to their heuristic value, steering early
       exploration without overriding learned statistics.
    """
 
    def __init__(
        self,
        time_limit: float = 1.0,
        iterations: Optional[int] = None,
        c: float = math.sqrt(2),
        prior_weight: float = 2.0,
    ):
        super().__init__(time_limit=time_limit, iterations=iterations, c=c)
        self.prior_weight = prior_weight
 
    # ------------------------------------------------------------------
    # Heuristic helpers
    # ------------------------------------------------------------------
 
    @staticmethod
    def _score_move(state, move: Tuple[str, int], player: int) -> float:
        """
        Quick heuristic score for a (move_type, col) from *player*'s view.
 
        Priority (highest first):
          5 — immediate win
          4 — block opponent's immediate win
          3 — create a 3-in-a-row threat
          2 — block opponent 3-in-a-row threat
          1 — central column bonus
          0 — base
        """
        opponent = 3 - player
 
        # Simulate the move
        test = state.copy()
        test.make_move(move[0], move[1])
 
        # Immediate win
        if test.game_over and test.winner == player:
            return 5.0
 
        # Block immediate opponent win
        block_test = state.copy()
        # Check if opponent would win with the same slot
        opp_moves = state.get_legal_moves()
        for om in opp_moves:
            bt = state.copy()
            bt.current_player = opponent
            bt.make_move(om[0], om[1])
            if bt.game_over and bt.winner == opponent:
                if om == move:
                    return 4.0  # we're blocking that column
 
        # 3-in-a-row opportunity (simplified: count our streaks after move)
        score = 1.0
        # Central column bonus (columns 3,4 in 0-indexed 7-col board)
        if move[1] in (3, 2, 4):
            score += 0.5
        return score
 
    def _weighted_choice(self, state, moves: List[Tuple[str, int]],
                         player: int) -> Tuple[str, int]:
        weights = [max(self._score_move(state, m, player), 0.01) for m in moves]
        total = sum(weights)
        r = random.random() * total
        cumulative = 0.0
        for m, w in zip(moves, weights):
            cumulative += w
            if r <= cumulative:
                return m
        return moves[-1]
 
    # ------------------------------------------------------------------
    # Overrides
    # ------------------------------------------------------------------
 
    def _expand(self, node: MCTSNode) -> MCTSNode:
        """Expand + armazena prior heurístico sem inflar as visitas reais."""
        child = super()._expand(node)
    
        # Cálculo do score heurístico baseado no estado atual
        prior = self._score_move(node.state, child.move, child.player)
        
        # Em vez de inflar child.visits, guardamos o peso separadamente
        # Isso evita que o denominador do UCT cresça sem simulações reais
        child.prior_wins = prior * self.prior_weight
        
        return child
 
    def _simulate(self, node: MCTSNode) -> float:
        """Biased rollout using heuristic weights."""
        state = node.state.copy()
        mover = node.player
 
        while not state.game_over:
            moves = state.get_legal_moves()
            if not moves:
                break
            move = self._weighted_choice(state, moves, state.current_player)
            state.make_move(move[0], move[1])
 
        return self._outcome(state, mover)
 
 
# ===========================================================================
# Convenience factory
# ===========================================================================
 
def make_mcts(strategy: str = "standard", **kwargs) -> MCTS:
    """
    Factory function.
 
    strategy: "standard" | "rave" | "topk" | "heuristic"
    kwargs  : forwarded to the chosen class constructor.
    """
    mapping = {
        "standard":  MCTS,
        "rave":      MCTS_RAVE,
        "topk":      MCTSTopK,
        "heuristic": MCTSWithHeuristics,
    }
    cls = mapping.get(strategy)
    if cls is None:
        raise ValueError(f"Unknown strategy '{strategy}'. "
                         f"Choose from {list(mapping.keys())}")
    return cls(**kwargs)

## 4. Comparision of MCTS Strategies

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 4.  Round-Robin Tournament:  4 × MCTS strategies
# ══════════════════════════════════════════════════════════════════════════

N_GAMES    = 20    # games per matchup (10 as P1, 10 as P2)
ITERATIONS = 200   # MCTS iterations per move (reduced for feasibility)

strategies = {
    'Standard'  : MCTS(iterations=ITERATIONS),
    'RAVE'      : MCTS_RAVE(iterations=ITERATIONS),
    'Top-K'     : MCTSTopK(iterations=ITERATIONS, k=7),
    'Heuristic' : MCTSWithHeuristics(iterations=ITERATIONS),
}

# ── Single game runner ────────────────────────────────────────────────────
def run_game(agent1, agent2):
    """
    Play one complete game between two agents.
    Returns (winner, n_moves, p1_ms_list, p2_ms_list).
    winner = 1 | 2 | None (draw)
    """
    game = PopOutGame(rows=6, cols=7)
    agents = {1: agent1, 2: agent2}
    move_times = {1: [], 2: []}

    while not game.game_over:
        p  = game.current_player
        t0 = time.time()
        move = agents[p].choose_move(game)
        move_times[p].append((time.time() - t0) * 1000)
        if move is None:
            break
        game.make_move(move[0], move[1])

    return game.winner, len(game.move_history), move_times[1], move_times[2]


# ── Round-robin ───────────────────────────────────────────────────────────
def run_tournament(strategies: dict, n_games: int = N_GAMES) -> pd.DataFrame:
    names   = list(strategies.keys())
    records = []

    for i, name1 in enumerate(names):
        for name2 in names[i + 1:]:
            a1_wins = a2_wins = draws = 0
            a1_times, a2_times, game_lengths = [], [], []
            half = n_games // 2

            print(f'  {name1:12s} vs {name2:12s}  ({n_games} games)...',
                  end=' ', flush=True)

            for g in range(n_games):
                p1_is_a1 = g < half
                if p1_is_a1:
                    w, nm, t1, t2 = run_game(strategies[name1], strategies[name2])
                else:
                    w, nm, t1, t2 = run_game(strategies[name2], strategies[name1])

                game_lengths.append(nm)
                a1_times.extend(t1 if p1_is_a1 else t2)
                a2_times.extend(t2 if p1_is_a1 else t1)

                if w is None:
                    draws += 1
                elif (w == 1 and p1_is_a1) or (w == 2 and not p1_is_a1):
                    a1_wins += 1
                else:
                    a2_wins += 1

            total = a1_wins + a2_wins + draws
            print(f'done  [{name1} {a1_wins}W  {name2} {a2_wins}W  {draws}D]')

            records.append({
                'A1': name1, 'A2': name2,
                'A1_wins': a1_wins, 'A2_wins': a2_wins, 'Draws': draws,
                'A1_win%': round(a1_wins / total * 100, 1),
                'A2_win%': round(a2_wins / total * 100, 1),
                'Draw%'  : round(draws   / total * 100, 1),
                'A1_ms'  : round(np.mean(a1_times), 1) if a1_times else 0.0,
                'A2_ms'  : round(np.mean(a2_times), 1) if a2_times else 0.0,
                'Avg_moves': round(np.mean(game_lengths), 1),
            })

    return pd.DataFrame(records)


print(f'Tournament: {N_GAMES} games/pair  |  {ITERATIONS} MCTS iters/move')
print('=' * 62)
df_tournament = run_tournament(strategies, n_games=N_GAMES)

print('\n=== Full Results ===')
cols_show = ['A1','A2','A1_wins','A2_wins','Draws',
             'A1_win%','A2_win%','Draw%','A1_ms','A2_ms','Avg_moves']
print(df_tournament[cols_show].to_string(index=False))


## 5. Decision Tree Implementation

Train a Decision Tree to learn optimal strategies from game data.

In [5]:
# ══════════════════════════════════════════════════════════════════════════
# 5.1  State encoding helpers
# ══════════════════════════════════════════════════════════════════════════

FEATURE_NAMES = (
    [f'r{r}c{c}' for r in range(6) for c in range(7)]  # 42 board cells
    + ['current_player']                                 # whose turn
)
N_ACTIONS = 14  # 7 drop + 7 pop

def encode_move(move: Tuple[str, int]) -> int:
    """Encode (type, col) → integer label 0-13.
    drop col c → c      (0-6)
    pop  col c → c + 7  (7-13)
    """
    mtype, col = move
    return col if mtype == 'drop' else col + 7

def decode_move(label: int) -> Tuple[str, int]:
    """Inverse of encode_move."""
    return ('drop', label) if label < 7 else ('pop', label - 7)

def state_to_features(game: PopOutGame) -> List[int]:
    """Flat feature vector: 42 board cells (0/1/2) + current_player (1/2)."""
    return list(game.board.flatten().astype(int)) + [game.current_player]

print(f"Feature vector length : {len(FEATURE_NAMES)}")
print(f"Action space size     : {N_ACTIONS}  (0-6 drop, 7-13 pop)")
print(f"Example encoding      : drop col 3 → {encode_move(('drop',3))}, "
      f"pop col 3 → {encode_move(('pop',3))}")


Feature vector length : 43
Action space size     : 14  (0-6 drop, 7-13 pop)
Example encoding      : drop col 3 → 3, pop col 3 → 10


### 5.2  Dataset Generation

After the tournament the **winning MCTS variant** is identified from `df_tournament` and used to label every position. Each self-play game produces one `(state, best_MCTS_move)` pair per half-move. Both players are driven by the same agent so the dataset covers positions from both sides of the board.


In [ ]:
# ── Identify the winning MCTS from tournament results ────────────────────
mcts_names  = ['Standard', 'RAVE', 'Top-K', 'Heuristic']
mcts_scores = {ag: 0 for ag in mcts_names}

for _, row in df_tournament.iterrows():
    a1, a2 = row['A1'], row['A2']
    if a1 in mcts_names and a2 in mcts_names:
        mcts_scores[a1] += row['A1_wins']
        mcts_scores[a2] += row['A2_wins']

best_mcts_name = max(mcts_scores, key=mcts_scores.get)

print("MCTS win counts (inter-MCTS games only):")
for ag, sc in sorted(mcts_scores.items(), key=lambda x: -x[1]):
    print(f"  {ag:12s}  {sc} wins")
print(f"\nTournament winner → {best_mcts_name}")

# ── Configuration ────────────────────────────────────────────────────────
N_DATASET_GAMES = 200   # self-play games to generate the dataset
MCTS_ITER_DATA  = 200   # MCTS iterations per move during generation
#   200 iter × ~30 moves/game × 200 games ≈ 1.2 M simulations total.

# ── Build the data agent from the tournament winner ───────────────────────
_data_agent_map = {
    'Standard'  : MCTS,
    'RAVE'      : MCTS_RAVE,
    'Top-K'     : MCTSTopK,
    'Heuristic' : MCTSWithHeuristics,
}
data_agent = _data_agent_map[best_mcts_name](iterations=MCTS_ITER_DATA)
print(f"Data agent : {best_mcts_name}  ({MCTS_ITER_DATA} iterations/move)\n")

# ── Collection loop ───────────────────────────────────────────────────────
import time as _time

X_raw: List[List[int]] = []
y_raw: List[int]       = []

t0 = _time.time()
for g_idx in range(N_DATASET_GAMES):
    game = PopOutGame(rows=6, cols=7)
    while not game.game_over:
        # Record (state, best-MCTS move) BEFORE applying the move
        features = state_to_features(game)
        move     = data_agent.choose_move(game)
        if move is None:
            break
        X_raw.append(features)
        y_raw.append(encode_move(move))
        game.make_move(move[0], move[1])
    if (g_idx + 1) % 20 == 0:
        elapsed = _time.time() - t0
        print(f"  Game {g_idx+1:>3}/{N_DATASET_GAMES}  "
              f"| samples so far: {len(X_raw):>5}  "
              f"| elapsed: {elapsed:.1f}s")

print(f"\nDataset generated: {len(X_raw)} samples from {N_DATASET_GAMES} games")
print(f"Total time: {_time.time()-t0:.1f}s")


NameError: name 'MCTSWithHeuristics' is not defined

### 5.3  Dataset Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

X = np.array(X_raw, dtype=np.int8)
y = np.array(y_raw,  dtype=np.int8)

df_dataset = pd.DataFrame(X, columns=FEATURE_NAMES)
df_dataset['action'] = y
df_dataset['action_str'] = [str(decode_move(int(a))) for a in y]

print(f"Shape            : {X.shape}")
print(f"Unique actions   : {np.unique(y)}")
print(f"Action counts:")
action_counts = pd.Series(y).value_counts().sort_index()
for lbl, cnt in action_counts.items():
    print(f"  {str(decode_move(int(lbl))):20s}  {cnt:5d}  ({cnt/len(y)*100:.1f}%)")

# Plot action distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

action_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Action Distribution (0-6: drop, 7-13: pop)')
axes[0].set_xlabel('Action label')
axes[0].set_ylabel('Frequency')
axes[0].set_xticks(range(14))
axes[0].set_xticklabels(
    [f'D{c}' for c in range(7)] + [f'P{c}' for c in range(7)], rotation=45)

# Cell occupation heatmap (avg player value per cell)
board_mean = X[:, :42].mean(axis=0).reshape(6, 7)
im = axes[1].imshow(board_mean, cmap='RdYlGn', vmin=0, vmax=2, aspect='auto')
axes[1].set_title('Average cell occupation across dataset')
axes[1].set_xlabel('Column'); axes[1].set_ylabel('Row (0=top)')
plt.colorbar(im, ax=axes[1], label='0=empty  1=P1  2=P2')

plt.tight_layout()
plt.show()


### 5.4  Decision Tree Training (ID3 / Information Gain)

We train a `DecisionTreeClassifier` with `criterion='entropy'` — this is the ID3 criterion (information gain). Because the raw board flattened to 43 integer features is already categorical-compatible, no further encoding is needed.

We also tune `max_depth` via cross-validation to balance accuracy vs. over-fitting, and explore the trade-off between tree depth and move quality.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pickle, warnings
warnings.filterwarnings('ignore')

# ── Train / test split (stratified by action) ────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

# ── Depth search via 5-fold CV ────────────────────────────────────────────
depths    = [3, 5, 8, 12, 18, 25, None]
cv_scores = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nDepth  |  CV Accuracy (mean ± std)")
print("-" * 38)
for d in depths:
    dt = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=42)
    scores = cross_val_score(dt, X_train, y_train, cv=skf, scoring='accuracy')
    cv_scores.append(scores.mean())
    label = str(d) if d else 'None'
    print(f"  {label:>4s}   |  {scores.mean():.4f} ± {scores.std():.4f}")

best_depth = depths[int(np.argmax(cv_scores))]
print(f"\nBest depth: {best_depth}")

# ── Final model at best depth ─────────────────────────────────────────────
dt_id3 = DecisionTreeClassifier(
    criterion='entropy', max_depth=best_depth, random_state=42
)
dt_id3.fit(X_train, y_train)

y_pred = dt_id3.predict(X_test)
print(f"\nTest accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Tree nodes    : {dt_id3.tree_.node_count}")
print(f"Tree leaves   : {dt_id3.get_n_leaves()}")


In [ ]:
# ── Depth vs accuracy plot ───────────────────────────────────────────────
depth_labels  = [str(d) if d else 'None' for d in depths]
train_accs = []
test_accs  = []

for d in depths:
    dt_tmp = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=42)
    dt_tmp.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt_tmp.predict(X_train)))
    test_accs.append (accuracy_score(y_test,  dt_tmp.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 4))
x_ticks = range(len(depths))
ax.plot(x_ticks, train_accs, 'o-', label='Train accuracy', color='steelblue')
ax.plot(x_ticks, test_accs,  's--', label='Test accuracy',  color='tomato')
ax.plot(x_ticks, cv_scores,  '^:', label='CV accuracy',     color='seagreen')
ax.axvline(x=int(np.argmax(cv_scores)), color='gray', linestyle=':', alpha=.7,
           label=f'Best depth = {best_depth}')
ax.set_xticks(x_ticks)
ax.set_xticklabels(depth_labels)
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('ID3 Decision Tree — Depth vs Accuracy')
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Classification report ────────────────────────────────────────────────
action_labels = [str(decode_move(i)) for i in range(N_ACTIONS)]
# Only report on classes present in test set
present = sorted(set(y_test))
print("Classification Report (actions present in test set):")
print(classification_report(
    y_test, y_pred,
    labels=present,
    target_names=[action_labels[i] for i in present],
    zero_division=0
))

# ── Confusion matrix heatmap ─────────────────────────────────────────────
import seaborn as sns
cm = confusion_matrix(y_test, y_pred, labels=present)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[action_labels[i] for i in present],
    yticklabels=[action_labels[i] for i in present],
    ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ID3 Decision Tree')
plt.tight_layout()
plt.show()


### 5.5  Decision Tree Visualisation

In [ ]:
# Top-level text view of the tree (first 4 levels)
print(export_text(dt_id3, feature_names=FEATURE_NAMES, max_depth=4))

# Graphical plot of the shallow subtree (max_depth=3 for readability)
fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(
    dt_id3, max_depth=3,
    feature_names=FEATURE_NAMES,
    class_names=[str(decode_move(i)) for i in range(N_ACTIONS)],
    filled=True, rounded=True,
    impurity=True, proportion=False,
    ax=ax, fontsize=7
)
ax.set_title('ID3 Decision Tree (top 3 levels)')
plt.tight_layout()
plt.savefig('dt_id3_tree.png', dpi=150, bbox_inches='tight')
plt.show()
print("Tree visualisation saved to dt_id3_tree.png")


In [ ]:
# ── Top feature importances (ID3 information-gain based) ─────────────────
importances = pd.Series(dt_id3.feature_importances_, index=FEATURE_NAMES)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(9, 5))
top20.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Information-gain importance')
ax.set_title('Top 20 Most Important Features (ID3 Decision Tree)')
ax.grid(axis='x', alpha=.3)
plt.tight_layout()
plt.show()

print("Top 10 features:")
for fname, imp in top20.head(10).items():
    print(f"  {fname:12s}  {imp:.5f}")


### 5.6  DecisionTreeAgent — Playing with the Trained Tree

Wrap the ID3 tree in the same `choose_move(game)` interface as the MCTS agents so it can be dropped directly into `run_game` / `run_tournament`. The tree predicts the *best action label*; if that action happens to be illegal in the current state (the tree may predict a move that is blocked), we fall back to the highest-confidence **legal** action from the predicted probability distribution.

In [ ]:
class DecisionTreeAgent:
    """
    Game agent backed by a trained sklearn DecisionTreeClassifier.

    choose_move() mirrors the MCTS agent interface so it can be used
    in run_game() / run_tournament() without modification.

    Fallback strategy: if the tree's top-1 predicted action is illegal,
    we rank all 14 actions by predicted probability and return the
    highest-probability one that is currently legal.
    """

    def __init__(self, model: DecisionTreeClassifier):
        self.model = model
        self._classes = list(model.classes_)   # action labels seen during training

    def choose_move(self, game: PopOutGame) -> Optional[Tuple[str, int]]:
        legal = game.get_legal_moves()
        if not legal:
            return None

        features = np.array([state_to_features(game)], dtype=np.int8)
        proba    = self.model.predict_proba(features)[0]   # shape (n_classes,)

        # Build a full probability array indexed 0-13
        full_proba = np.zeros(N_ACTIONS, dtype=float)
        for cls_idx, cls_label in enumerate(self._classes):
            full_proba[cls_label] = proba[cls_idx]

        # Rank legal actions by predicted probability
        legal_encoded = {encode_move(m): m for m in legal}
        best_label = max(legal_encoded.keys(),
                         key=lambda lbl: full_proba[lbl])
        return legal_encoded[best_label]


dt_agent = DecisionTreeAgent(dt_id3)

# Quick sanity check — one full game DT vs DT
game_check = PopOutGame()
while not game_check.game_over:
    m = dt_agent.choose_move(game_check)
    if m is None: break
    game_check.make_move(*m)
print("DT self-play check:", game_check.get_status())
print(f"Game lasted {len(game_check.move_history)} moves")


In [ ]:
# ── Persist the trained ID3 model ────────────────────────────────────────
import pickle

DT_MODEL_PATH = 'dt_id3_model.pkl'
with open(DT_MODEL_PATH, 'wb') as fh:
    pickle.dump(dt_id3, fh)
print(f"ID3 model saved → {DT_MODEL_PATH}")

# Reload test
with open(DT_MODEL_PATH, 'rb') as fh:
    dt_id3_loaded = pickle.load(fh)
assert accuracy_score(y_test, dt_id3_loaded.predict(X_test)) == accuracy_score(y_test, y_pred)
print("Reload verification: OK")


## 6. Visualize Game Trees and Results

In [ ]:
# ── Win-rate heatmap across all matchups ─────────────────────────────────
agent_names = list(strategies.keys())
n = len(agent_names)
win_matrix = pd.DataFrame(np.full((n, n), np.nan),
                           index=agent_names, columns=agent_names)

for _, row in df_tournament.iterrows():
    a1, a2 = row['A1'], row['A2']
    win_matrix.loc[a1, a2] = row['A1_win%']
    win_matrix.loc[a2, a1] = row['A2_win%']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Win-rate heatmap
sns.heatmap(win_matrix.astype(float), annot=True, fmt='.1f',
            cmap='RdYlGn', vmin=0, vmax=100, linewidths=.5,
            ax=axes[0], cbar_kws={'label': 'Win %'})
axes[0].set_title('Win % (row agent vs column agent)')

# Average move time comparison
time_data = {}
for _, row in df_tournament.iterrows():
    for ag, col in [(row['A1'], 'A1_ms'), (row['A2'], 'A2_ms')]:
        time_data.setdefault(ag, []).append(row[col])
avg_times = {ag: np.mean(v) for ag, v in time_data.items()}
pd.Series(avg_times).sort_values().plot(
    kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_xlabel('Average ms per move')
axes[1].set_title('Avg Move Time per Agent')
axes[1].grid(axis='x', alpha=.3)

plt.tight_layout()
plt.show()

# ── Overall win-rate per agent (sum across all matchups) ──────────────────
print("\nOverall performance summary:")
total_wins = {ag: 0 for ag in agent_names}
total_games= {ag: 0 for ag in agent_names}
for _, row in df_tournament.iterrows():
    for ag, wcol, gcol in [
        (row['A1'], 'A1_wins', 'A1_wins'),
        (row['A2'], 'A2_wins', 'A2_wins'),
    ]:
        total_wins[ag]  += row[wcol]
        total_games[ag] += row['A1_wins'] + row['A2_wins'] + row['Draws']
for ag in agent_names:
    g = total_games[ag]
    w = total_wins[ag]
    print(f"  {ag:12s}  {w:3d} wins / {g:3d} games  ({w/g*100:.1f}%)")


## 7. Summary and Conclusions

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 7.  Identify & Persist the Best MCTS Agent
# ══════════════════════════════════════════════════════════════════════════

# ── Identify the best MCTS strategy from tournament results ──────────────
mcts_names = ['Standard', 'RAVE', 'Top-K', 'Heuristic']
mcts_scores = {ag: 0 for ag in mcts_names}

for _, row in df_tournament.iterrows():
    a1, a2 = row['A1'], row['A2']
    # Only count MCTS vs MCTS matchups for the ranking
    if a1 in mcts_names and a2 in mcts_names:
        mcts_scores[a1] += row['A1_wins']
        mcts_scores[a2] += row['A2_wins']

best_mcts_name = max(mcts_scores, key=mcts_scores.get)
print("MCTS win counts (inter-MCTS games only):")
for ag, sc in sorted(mcts_scores.items(), key=lambda x: -x[1]):
    print(f"  {ag:12s}  {sc} wins")
print(f"\nBest MCTS strategy: {best_mcts_name}")

# ── Build the production-strength best agent ──────────────────────────────
# Use more iterations than tournament (better quality for actual gameplay)
PRODUCTION_ITER = 500
best_mcts_map = {
    'Standard'  : MCTS(iterations=PRODUCTION_ITER),
    'RAVE'      : MCTS_RAVE(iterations=PRODUCTION_ITER),
    'Top-K'     : MCTSTopK(iterations=PRODUCTION_ITER, k=7),
    'Heuristic' : MCTSWithHeuristics(iterations=PRODUCTION_ITER),
}
best_mcts_agent = best_mcts_map[best_mcts_name]

# ── Persist with pickle ───────────────────────────────────────────────────
import pickle

BEST_MCTS_PATH = 'best_mcts_agent.pkl'
with open(BEST_MCTS_PATH, 'wb') as fh:
    pickle.dump(best_mcts_agent, fh)
print(f"Best MCTS agent saved → {BEST_MCTS_PATH}")

# ── Reload & verify ───────────────────────────────────────────────────────
with open(BEST_MCTS_PATH, 'rb') as fh:
    best_mcts_loaded = pickle.load(fh)

game_verify = PopOutGame()
game_verify.make_move('drop', 3)
test_move = best_mcts_loaded.choose_move(game_verify)
print(f"Reload verification — suggested move: {test_move}  ✓")

print()
print("=" * 55)
print(f" Best MCTS : {best_mcts_name} ({PRODUCTION_ITER} iterations)")
print(f" Saved to  : {BEST_MCTS_PATH}")
print(f" ID3 Tree  : dt_id3_model.pkl")
print("=" * 55)
print()
print("Usage in interface.py / game loop:")
print("  import pickle")
print(f"  with open('{BEST_MCTS_PATH}', 'rb') as f:")
print("      agent = pickle.load(f)")
print("  move = agent.choose_move(game)")


## 8. API Execution/Visualization (Best MCTS vs ID3)